# Does adding ICD features improve the readmission model?

This trains the **current model** and an **ICD-enriched model** on the same split, with the same
hyperparameters, and compares them honestly.

## What is already in the model

A correction first, because an earlier version of the project documentation got this wrong. The
94-feature model **already contains all 17 Charlson condition flags** —
`congestive_heart_failure`, `renal_disease`, `metastatic_cancer` and the rest — as individual
columns, plus `charlson_score` and `n_diagnoses`. Nineteen of the 94 features are
diagnosis-derived.

So this notebook does **not** add Charlson flags. It adds what genuinely is not there:

| Feature group | Why it might help |
|---|---|
| `principal_chapter` | the coder's judgement about **why** the patient was admitted, currently discarded. Readmission rates across its chapters span 0.53× to 1.69× |
| Chapter flags | only the informative ones. Chapters covering two thirds of admissions carry no signal — circulatory is 1.04× — so adding all 20 would be mostly noise |
| List shape | `n_chapters`, `n_categories`, `n_status_codes`: how many body systems are in play, how much of the list is social rather than medical. Clean monotonic signal, 0.77× → 1.44× |
| `dx_sepsis`, `has_complication` | conditions with real signal that Charlson does not cover at all |
| Combinations | cardiorenal, diabetes+renal, cancer+organ — each measurably stronger than its parts |

Deliberately **excluded**, on the evidence from `mimic_icd_feature_engineering.ipynb`:
`dx_atrial_fibrillation` (1.08× alone, and the AF+heart-failure combination scored *below* heart
failure by itself), and the chapter flags for circulatory, health status, endocrine and
musculoskeletal.

## What counts as an improvement

AUC-ROC is the least useful number here — the outcome is imbalanced, so **AUC-PR** matters more.
And because the whole ROI layer multiplies by a predicted probability, **calibration** matters as
much as discrimination: a model that ranks better but is systematically over-confident is worse
for this application, not better.

A plausible outcome is a small AUC gain and a large explainability gain. That is still worth
shipping, but it should be reported as what it is.

> **Attach two inputs.** A MIMIC-IV dataset (for `diagnoses_icd`), and `phase1_matrix.parquet`
> uploaded as a Kaggle dataset. The matrix is in the repo at
> `data/mimic/model/results/phase1_matrix.parquet`.

## 0. Load the matrix and the raw diagnoses

In [ ]:
import os, re, glob, json, warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

SEARCH_ROOTS = ["/kaggle/input", "../input", "./data", "."]

def find_file(*names, exts=("parquet", "csv", "csv.gz")):
    hits = []
    for root in SEARCH_ROOTS:
        if not os.path.isdir(root):
            continue
        for name in names:
            for ext in exts:
                hits += glob.glob(f"{root}/**/{name}.{ext}", recursive=True)
    return max(set(hits), key=os.path.getsize) if hits else None

matrix_path = find_file("phase1_matrix")
dx_path = find_file("diagnoses_icd", "sample_diagnoses_icd", exts=("csv", "csv.gz", "parquet"))

print("phase1_matrix :", matrix_path or "NOT FOUND")
print("diagnoses_icd :", dx_path or "NOT FOUND")

if not matrix_path:
    raise SystemExit(
        "Upload phase1_matrix.parquet as a Kaggle dataset and attach it with + Add Input.\n"
        "It lives in the repo at "
        "data/mimic/model/results/phase1_matrix.parquet"
    )
if not dx_path:
    raise SystemExit("Attach a MIMIC-IV dataset so diagnoses_icd can be read.")

X = pd.read_parquet(matrix_path)
print(f"\nmatrix: {X.shape[0]:,} admissions x {X.shape[1]} columns, "
      f"{X.subject_id.nunique():,} patients")
print(f"readmission rate: {X.readmit_30d.mean():.2%}")

# The 94 features, exactly as the shipped model defines them.
CATEGORICALS = ["gender", "insurance", "admission_type", "marital_status", "race"]
FEATURES = [c for c in X.columns if c not in ("subject_id", "hadm_id", "readmit_30d")]
print(f"baseline features: {len(FEATURES)}")

charlson_flags = [c for c in FEATURES if c in (
    "myocardial_infarction", "congestive_heart_failure", "peripheral_vascular",
    "cerebrovascular", "dementia", "chronic_pulmonary", "rheumatic", "peptic_ulcer",
    "mild_liver", "diabetes_uncomplicated", "diabetes_complicated", "hemiplegia",
    "renal_disease", "malignancy", "severe_liver", "metastatic_cancer", "hiv_aids")]
print(f"Charlson flags already present: {len(charlson_flags)}/17  <- not re-added below")

## 1. Build the ICD features

Only the ones that are not already in the matrix, and only the ones the signal report marked as
worth carrying.

In [ ]:
t0 = time.time()
dx = (pd.read_parquet(dx_path, columns=["hadm_id", "seq_num", "icd_code", "icd_version"])
      if dx_path.endswith("parquet") else
      pd.read_csv(dx_path, usecols=["hadm_id", "seq_num", "icd_code", "icd_version"]))

dx = dx[dx.hadm_id.isin(set(X.hadm_id))]
dx["icd_code"] = dx.icd_code.astype(str).str.upper().str.replace(".", "", regex=False).str.strip()
dx["icd_version"] = pd.to_numeric(dx.icd_version, errors="coerce").astype("Int64")
print(f"{len(dx):,} diagnosis rows covering {dx.hadm_id.nunique():,} of "
      f"{X.hadm_id.nunique():,} admissions in the matrix")

ICD10_CHAPTERS = [
    ("A00","B99","infectious"), ("C00","D49","neoplasm"), ("D50","D89","blood"),
    ("E00","E89","endocrine"), ("F01","F99","mental"), ("G00","G99","nervous"),
    ("H00","H59","nervous"), ("H60","H95","nervous"), ("I00","I99","circulatory"),
    ("J00","J99","respiratory"), ("K00","K95","digestive"), ("L00","L99","skin"),
    ("M00","M99","musculoskeletal"), ("N00","N99","genitourinary"),
    ("O00","O9A","obstetric"), ("P00","P96","perinatal"), ("Q00","Q99","congenital"),
    ("R00","R99","symptoms"), ("S00","T88","injury"), ("U00","U85","special"),
    ("V00","Y99","external_cause"), ("Z00","Z99","health_status"),
]
ICD9_CHAPTERS = [
    (1,139,"infectious"), (140,239,"neoplasm"), (240,279,"endocrine"), (280,289,"blood"),
    (290,319,"mental"), (320,389,"nervous"), (390,459,"circulatory"), (460,519,"respiratory"),
    (520,579,"digestive"), (580,629,"genitourinary"), (630,679,"obstetric"),
    (680,709,"skin"), (710,739,"musculoskeletal"), (740,759,"congenital"),
    (760,779,"perinatal"), (780,799,"symptoms"), (800,999,"injury"),
]

def chapter_of(code, version):
    c = str(code).strip().upper()
    if not c:
        return "unclassified"
    if version == 9:
        if c[0] == "E":
            return "external_cause"
        if c[0] == "V":
            return "health_status"
        try:
            head = int(c[:3])
        except ValueError:
            return "unclassified"
        for lo, hi, name in ICD9_CHAPTERS:
            if lo <= head <= hi:
                return name
        return "unclassified"
    head = c[:3]
    for lo, hi, name in ICD10_CHAPTERS:
        if lo <= head <= hi:
            return name
    return "unclassified"

dx["chapter"] = [chapter_of(c, v) for c, v in zip(dx.icd_code, dx.icd_version)]
dx["category"] = dx.icd_code.str.slice(0, 3)
print(f"chapters mapped in {time.time() - t0:.0f}s")

In [ ]:
# Only the chapters that carry signal. The ones covering two thirds of
# admissions - circulatory 1.04x, health_status 1.06x, endocrine 1.06x,
# musculoskeletal 1.02x - are omitted: a flag on most of the cohort separates
# nothing and only adds a column for the model to split on by accident.
KEEP_CHAPTERS = ["neoplasm", "infectious", "blood", "obstetric", "mental",
                 "digestive", "genitourinary", "skin", "respiratory", "nervous"]

feat = pd.DataFrame(index=pd.Index(X.hadm_id.unique(), name="hadm_id"))

flags = (dx[dx.chapter.isin(KEEP_CHAPTERS)].assign(one=1)
           .pivot_table(index="hadm_id", columns="chapter", values="one",
                        aggfunc="max", fill_value=0))
flags.columns = [f"chap_{c}" for c in flags.columns]
feat = feat.join(flags.astype("int8")).fillna(0)

g = dx.groupby("hadm_id")
feat["n_chapters"] = g.chapter.nunique()
feat["n_categories"] = g.category.nunique()
feat["n_status_codes"] = dx[dx.chapter.eq("health_status")].groupby("hadm_id").size()
feat["n_symptom_codes"] = dx[dx.chapter.eq("symptoms")].groupby("hadm_id").size()

# In-hospital complications, as a BINARY. The count is zero for ~88% of
# admissions, which is why the earlier notebook's quantile banding collapsed to
# a single bin and never produced a verdict on it.
comp = dx[((dx.icd_version.eq(10)) & dx.icd_code.str.match(r"^T8[0-8]")) |
          ((dx.icd_version.eq(9)) & dx.icd_code.str.match(r"^99[6-9]"))]
feat["has_complication"] = feat.index.isin(comp.hadm_id.unique()).astype("int8")

# Sepsis: real signal, and absent from Charlson entirely.
sep = dx[((dx.icd_version.eq(10)) & dx.icd_code.str.match(r"^(A40|A41|R652)")) |
         ((dx.icd_version.eq(9)) & dx.icd_code.str.match(r"^(038|9959[12])"))]
feat["dx_sepsis"] = feat.index.isin(sep.hadm_id.unique()).astype("int8")

# The chapter of the principal diagnosis. Categorical, and the strongest of the
# new features by univariate spread.
principal = (dx[dx.seq_num.eq(1)].sort_values("icd_version")
               .drop_duplicates("hadm_id").set_index("hadm_id"))
feat["principal_chapter"] = principal.chapter

feat[["n_chapters", "n_categories", "n_status_codes", "n_symptom_codes"]] = (
    feat[["n_chapters", "n_categories", "n_status_codes", "n_symptom_codes"]].fillna(0))
feat["principal_chapter"] = feat.principal_chapter.fillna("missing").astype(str)

print(f"{feat.shape[1]} ICD features built for {len(feat):,} admissions")
feat.head(3)

In [ ]:
# Combinations, built from the Charlson flags ALREADY in the matrix rather than
# recomputed - so they cannot drift from what the model already sees.
base = X.set_index("hadm_id")
combos = pd.DataFrame(index=feat.index)
combos["combo_cardiorenal"] = ((base.congestive_heart_failure.eq(1)) &
                               (base.renal_disease.eq(1))).astype("int8")
combos["combo_diabetes_renal"] = (((base.diabetes_complicated.eq(1)) |
                                   (base.diabetes_uncomplicated.eq(1))) &
                                  (base.renal_disease.eq(1))).astype("int8")
combos["combo_cardiopulmonary"] = ((base.congestive_heart_failure.eq(1)) &
                                   (base.chronic_pulmonary.eq(1))).astype("int8")
combos["combo_cancer_plus_organ"] = (((base.malignancy.eq(1)) |
                                      (base.metastatic_cancer.eq(1))) &
                                     ((base.renal_disease.eq(1)) |
                                      (base.congestive_heart_failure.eq(1)))).astype("int8")
feat = feat.join(combos.reindex(feat.index).fillna(0))

NEW_FEATURES = list(feat.columns)
NEW_CATEGORICALS = ["principal_chapter"]
print(f"{len(NEW_FEATURES)} new features:\n  " + "\n  ".join(NEW_FEATURES))

XE = X.merge(feat.reset_index(), on="hadm_id", how="left", validate="one_to_one")
for c in NEW_FEATURES:
    if c not in NEW_CATEGORICALS:
        XE[c] = pd.to_numeric(XE[c], errors="coerce").fillna(0)
XE["principal_chapter"] = XE.principal_chapter.fillna("missing").astype("category")
print(f"\nenriched matrix: {XE.shape[0]:,} x {XE.shape[1]}")
assert len(XE) == len(X), "the join changed the row count"

## 2. The split

Identical to Phase 1: `GroupShuffleSplit` on `subject_id`, so no patient appears in more than one
fold. That matters more than usual here — 45% of patients have several admissions, and splitting
by row would let the model memorise a patient in training and be graded on them in test.

Both models get **exactly the same rows**, so any difference is the features and nothing else.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

y = XE.readmit_30d.values
groups = XE.subject_id.values

gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
trainval_idx, test_idx = next(gss.split(XE, y, groups))

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
tr_rel, cal_rel = next(gss2.split(XE.iloc[trainval_idx], y[trainval_idx], groups[trainval_idx]))
train_idx = trainval_idx[tr_rel]
calib_idx = trainval_idx[cal_rel]

for name, idx in [("train", train_idx), ("calibration", calib_idx), ("test", test_idx)]:
    print(f"  {name:<12} rows={len(idx):>8,}  patients={pd.Series(groups[idx]).nunique():>7,}"
          f"  positives={y[idx].mean():.2%}")

assert not (set(groups[train_idx]) & set(groups[test_idx])), "patient leakage across folds"
assert not (set(groups[calib_idx]) & set(groups[test_idx])), "patient leakage across folds"
print("\n  no patient appears in more than one fold")

## 3. Train both models

Same hyperparameters as the shipped model, so the baseline here reproduces it rather than
approximating it: `HistGradientBoostingClassifier` with native categorical support, then
`CalibratedClassifierCV` with isotonic regression fitted on the held-out calibration fold.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV

HGB_PARAMS = dict(max_iter=400, learning_rate=0.06, max_leaf_nodes=31,
                  min_samples_leaf=40, l2_regularization=1.0,
                  early_stopping=True, validation_fraction=0.15, random_state=42)

def fit_model(features, categoricals, label):
    Xf = XE[features].copy()
    for c in categoricals:
        Xf[c] = Xf[c].astype("category")
    cat_mask = [c in categoricals for c in features]

    t = time.time()
    hgb = HistGradientBoostingClassifier(categorical_features=cat_mask, **HGB_PARAMS)
    hgb.fit(Xf.iloc[train_idx], y[train_idx])

    method = "isotonic" if len(calib_idx) >= 2000 else "sigmoid"
    cal = CalibratedClassifierCV(hgb, method=method, cv="prefit")
    cal.fit(Xf.iloc[calib_idx], y[calib_idx])
    print(f"  {label:<28} {len(features):>3} features   {time.time() - t:5.0f}s")
    return {"raw": hgb, "cal": cal, "features": features,
            "categoricals": categoricals, "X": Xf}

print("fitting ...")
BASE = fit_model(FEATURES, CATEGORICALS, "baseline (current model)")
ENR  = fit_model(FEATURES + NEW_FEATURES, CATEGORICALS + NEW_CATEGORICALS,
                 "enriched (+ ICD features)")

## 4. The comparison

Four numbers, and only two of them are about ranking:

- **AUC-ROC** — ranking, and the least informative here because the outcome is imbalanced.
- **AUC-PR** — ranking, weighted toward the positives that actually matter. Compare against the
  base rate, not against 0.5.
- **Brier score** — calibration and discrimination together. **Lower is better.**
- **Calibration slope / intercept** — whether a predicted 30% really means 30%. The ROI layer
  multiplies by this probability, so a model that ranks better while being over-confident is
  worse for this application.

In [ ]:
from sklearn.metrics import (roc_auc_score, average_precision_score, brier_score_loss,
                             precision_recall_curve, roc_curve)
from sklearn.calibration import calibration_curve

yte = y[test_idx]
base_rate = yte.mean()

def evaluate(m, label):
    p = m["cal"].predict_proba(m["X"].iloc[test_idx])[:, 1]
    frac_pos, mean_pred = calibration_curve(yte, p, n_bins=10, strategy="quantile")
    slope = np.polyfit(mean_pred, frac_pos, 1)[0]
    return {
        "model": label,
        "AUC-ROC": roc_auc_score(yte, p),
        "AUC-PR": average_precision_score(yte, p),
        "Brier": brier_score_loss(yte, p),
        "calib_slope": slope,
        "mean_pred": p.mean(),
        "_p": p,
    }

res_base = evaluate(BASE, "baseline (94 features)")
res_enr  = evaluate(ENR,  f"enriched (+{len(NEW_FEATURES)} ICD features)")

comp = pd.DataFrame([{k: v for k, v in r.items() if not k.startswith("_")}
                     for r in (res_base, res_enr)]).set_index("model")
comp.loc["difference"] = comp.loc[comp.index[1]] - comp.loc[comp.index[0]]

print(f"test set: {len(yte):,} admissions, {yte.mean():.2%} readmitted")
print(f"a perfectly uninformative AUC-PR would be {base_rate:.3f}\n")
print(comp.round(4).to_string())
print("\nAUC-ROC and AUC-PR: higher is better.  Brier: LOWER is better.")
print("calib_slope: 1.0 is perfect; below 1 means over-confident predictions.")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))

for res, colour in ((res_base, "#9aa5b1"), (res_enr, "#0F2A4A")):
    fpr, tpr, _ = roc_curve(yte, res["_p"])
    axes[0].plot(fpr, tpr, color=colour,
                 label=f"{res['model']}  {res['AUC-ROC']:.3f}")
    prec, rec, _ = precision_recall_curve(yte, res["_p"])
    axes[1].plot(rec, prec, color=colour,
                 label=f"{res['model']}  {res['AUC-PR']:.3f}")
    frac_pos, mean_pred = calibration_curve(yte, res["_p"], n_bins=10, strategy="quantile")
    axes[2].plot(mean_pred, frac_pos, "o-", color=colour, label=res["model"])

axes[0].plot([0, 1], [0, 1], "--", color="#C0392B", lw=1)
axes[0].set(xlabel="false positive rate", ylabel="true positive rate", title="ROC")
axes[1].axhline(base_rate, ls="--", color="#C0392B", lw=1)
axes[1].set(xlabel="recall", ylabel="precision", title="Precision-Recall")
axes[2].plot([0, 1], [0, 1], "--", color="#C0392B", lw=1)
axes[2].set(xlabel="predicted probability", ylabel="observed rate",
            title="Calibration", xlim=(0, 0.8), ylim=(0, 0.8))
for ax in axes:
    ax.legend(fontsize=8, loc="best")
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
plt.tight_layout()

In [ ]:
# What the difference means in patients, not decimals.
#
# At the shipped operating threshold, how many of the same number of flagged
# patients actually come back? That is the number a care team feels.
THRESHOLD = 0.2243   # the operating threshold in the shipped bundle

rows = []
for res in (res_base, res_enr):
    p = res["_p"]
    flagged = p >= THRESHOLD
    # Also compare at a fixed BUDGET: if the team can only work 500 patients,
    # how many readmissions are in the top 500 by risk?
    for budget in (500, 1000):
        top = np.argsort(-p)[:budget]
        rows.append({"model": res["model"], "view": f"top {budget} by risk",
                     "caught": int(yte[top].sum()),
                     "precision": yte[top].mean()})
    rows.append({"model": res["model"], "view": f"threshold {THRESHOLD:.2f}",
                 "flagged": int(flagged.sum()),
                 "caught": int(yte[flagged].sum()),
                 "precision": yte[flagged].mean() if flagged.any() else 0.0,
                 "recall": yte[flagged].sum() / yte.sum()})

ops = pd.DataFrame(rows)
print("OPERATIONAL VIEW\n")
print(ops.fillna("").round(4).to_string(index=False))
print(f"\ntotal readmissions in the test set: {int(yte.sum()):,}")

## 5. Did the new features get used, and for what?

A feature can improve AUC by a rounding error and still be worth adding if it makes the
explanation on a driver card specific — that is a goal in its own right for this project. The
question here is whether the model reaches for the new features at all, and which ones.

In [ ]:
import shap

# SHAP on a sample: the full test fold is 59,000 rows and the ranking is stable
# long before that.
SAMPLE = 4000
idx = np.random.RandomState(42).choice(test_idx, min(SAMPLE, len(test_idx)), replace=False)

Xs = ENR["X"].iloc[idx].copy()
for c in ENR["categoricals"]:
    Xs[c] = Xs[c].cat.codes          # TreeExplainer needs numeric input
explainer = shap.TreeExplainer(ENR["raw"])
sv = np.array(explainer.shap_values(Xs))
if sv.ndim == 3:
    sv = sv[:, :, 1]

importance = (pd.DataFrame({"feature": ENR["features"],
                            "mean_abs_shap": np.abs(sv).mean(axis=0)})
              .sort_values("mean_abs_shap", ascending=False)
              .reset_index(drop=True))
importance["is_new"] = importance.feature.isin(NEW_FEATURES)
importance["rank"] = importance.index + 1

new_share = importance.loc[importance.is_new, "mean_abs_shap"].sum() / importance.mean_abs_shap.sum()
print(f"the {len(NEW_FEATURES)} new features carry {new_share:.1%} of total attribution\n")
print("WHERE EACH NEW FEATURE RANKS AMONG ALL " f"{len(ENR['features'])}:")
print(importance[importance.is_new][["rank", "feature", "mean_abs_shap"]]
      .to_string(index=False))
print("\nTOP 20 OVERALL (new features marked):")
top20 = importance.head(20).copy()
top20["feature"] = np.where(top20.is_new, "* " + top20.feature, "  " + top20.feature)
print(top20[["rank", "feature", "mean_abs_shap"]].to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
top = importance.head(22).iloc[::-1]
ax.barh(top.feature, top.mean_abs_shap,
        color=np.where(top.is_new, "#C0392B", "#0F2A4A"))
ax.set_xlabel("mean |SHAP| — average contribution to a prediction")
ax.set_title("Feature importance, enriched model  (red = newly added)")
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
plt.tight_layout()

## 6. Export

Three artefacts:

- `phase1_model_icd.joblib` — the enriched bundle in the same shape the application already
  loads: `{model, features, categoricals, threshold}`. Dropping it into
  `data/mimic/model/results/` and pointing the loader at it is the
  whole deployment step.
- `phase1_matrix_icd.parquet` — the enriched matrix, so scoring can be reproduced without
  rebuilding the features.
- `icd_training_comparison.json` — the numbers, for pasting back into the project.

In [ ]:
import joblib

OUT = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."

bundle = {
    "model": ENR["cal"],
    "features": ENR["features"],
    "categoricals": ENR["categoricals"],
    # Kept from the shipped bundle so band thresholds do not silently move. If
    # the enriched model shifts the score distribution, re-derive the bands with
    # scripts/load_mimic_to_mongo.py rather than assuming this still fits.
    "threshold": 0.22429906542056074,
}
joblib.dump(bundle, f"{OUT}/phase1_model_icd.joblib")

keep = ["subject_id", "hadm_id", "readmit_30d"] + ENR["features"]
XE[keep].to_parquet(f"{OUT}/phase1_matrix_icd.parquet", index=False)

summary = {
    "cohort": {"admissions": int(len(XE)), "patients": int(XE.subject_id.nunique()),
               "readmission_rate": float(y.mean())},
    "test_set": {"admissions": int(len(test_idx)), "readmission_rate": float(yte.mean())},
    "baseline": {k: float(v) for k, v in res_base.items() if not k.startswith("_")
                 and k != "model"},
    "enriched": {k: float(v) for k, v in res_enr.items() if not k.startswith("_")
                 and k != "model"},
    "new_features": NEW_FEATURES,
    "new_feature_attribution_share": float(new_share),
    "new_feature_ranks": {r.feature: int(r["rank"])
                          for _, r in importance[importance.is_new].iterrows()},
}
with open(f"{OUT}/icd_training_comparison.json", "w") as fh:
    json.dump(summary, fh, indent=2)

for f in ("phase1_model_icd.joblib", "phase1_matrix_icd.parquet",
          "icd_training_comparison.json"):
    print(f"  {f:<34} {os.path.getsize(f'{OUT}/{f}'):>14,} bytes")

In [ ]:
# The verdict, in the form it should be reported back.
d_auc  = res_enr["AUC-ROC"] - res_base["AUC-ROC"]
d_pr   = res_enr["AUC-PR"]  - res_base["AUC-PR"]
d_brier = res_enr["Brier"]  - res_base["Brier"]

print("ICD FEATURE TRAINING - RESULT")
print(f"  cohort              : {len(XE):,} admissions, {XE.subject_id.nunique():,} patients")
print(f"  test set            : {len(test_idx):,} admissions, {yte.mean():.2%} readmitted")
print(f"  features            : {len(FEATURES)} -> {len(ENR['features'])}  "
      f"(+{len(NEW_FEATURES)})")
print()
print(f"  AUC-ROC   {res_base['AUC-ROC']:.4f} -> {res_enr['AUC-ROC']:.4f}   ({d_auc:+.4f})")
print(f"  AUC-PR    {res_base['AUC-PR']:.4f} -> {res_enr['AUC-PR']:.4f}   ({d_pr:+.4f})")
print(f"  Brier     {res_base['Brier']:.4f} -> {res_enr['Brier']:.4f}   ({d_brier:+.4f})"
      "   lower is better")
print(f"  calib     {res_base['calib_slope']:.3f} -> {res_enr['calib_slope']:.3f}"
      "   1.0 is perfect")
print()
print(f"  new features carry {new_share:.1%} of total SHAP attribution")
best_new = importance[importance.is_new].head(5)
for _, r in best_new.iterrows():
    print(f"    rank {int(r['rank']):>3}  {r.feature}")
print()

verdict = []
if d_pr > 0.002:
    verdict.append("AUC-PR improved meaningfully")
elif d_pr > 0:
    verdict.append("AUC-PR improved marginally")
else:
    verdict.append("AUC-PR did NOT improve")
if d_brier < -0.0005:
    verdict.append("calibration improved")
elif d_brier > 0.0005:
    verdict.append("calibration got WORSE - this matters, the ROI layer depends on it")
else:
    verdict.append("calibration unchanged")
if new_share > 0.05:
    verdict.append(f"the model does use the new features ({new_share:.0%} of attribution)")
else:
    verdict.append(f"the model barely uses the new features ({new_share:.1%} of attribution)")
print("  VERDICT: " + "; ".join(verdict) + ".")
print()
print("  Remember what a small AUC gain with a large explainability gain means: it is still")
print("  worth shipping if the driver cards become specific, but it should be reported as")
print("  that and not as a performance win.")